# FinTrust Digital Bank – Week 4

## Part A – Final Data & Analysis Review

### Objective

This notebook performs the final review of the FinTrust Data Analytics work completed during Weeks 1–3.

The review checks data quality, calculation accuracy, SQL results, Python reproducibility, KPI definitions and whether the final business findings are supported by evidence.

Any corrections or improvements identified during testing will be documented.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
customers = pd.read_csv("FinTrust_Customer_Clean Data.csv")
transactions = pd.read_csv("FinTrust_Transaction_Clean Data.csv")

In [3]:
print(customers.shape)
print(transactions.shape)

(1500, 12)
(12000, 12)


In [4]:
print(customers.shape)
print(transactions.shape)

(1500, 12)
(12000, 12)


## A2 – Final Data Quality Check

This check confirms that the final cleaned customer and transaction datasets contain the expected number of records, no duplicate IDs, and no unresolved missing values.

In [5]:
print("CUSTOMER DATA QUALITY")
print("Duplicate Customer IDs:", customers["Customer_ID"].duplicated().sum())
print("Total Missing Values:", customers.isnull().sum().sum())

print("\nTRANSACTION DATA QUALITY")
print("Duplicate Transaction IDs:", transactions["Transaction_ID"].duplicated().sum())
print("Total Missing Values:", transactions.isnull().sum().sum())

CUSTOMER DATA QUALITY
Duplicate Customer IDs: 0
Total Missing Values: 0

TRANSACTION DATA QUALITY
Duplicate Transaction IDs: 0
Total Missing Values: 192


## A3 – Referential Integrity Check

This test verifies that every transaction Customer_ID exists in the customer dataset and that the relationship between the two datasets remains valid.

In [6]:
unmatched_transactions = transactions[
    ~transactions["Customer_ID"].isin(customers["Customer_ID"])
]

print("Transactions with unmatched Customer IDs:", len(unmatched_transactions))

Transactions with unmatched Customer IDs: 0


## A4 – Transaction Amount Validation

This test checks whether any transaction contains a zero or negative amount and confirms the final transaction amount range.

In [7]:
print("Transactions with Amount <= 0:",
      (transactions["Amount_NGN"] <= 0).sum())

print("Minimum Transaction Amount:",
      transactions["Amount_NGN"].min())

print("Maximum Transaction Amount:",
      transactions["Amount_NGN"].max())

Transactions with Amount <= 0: 0
Minimum Transaction Amount: 100.17
Maximum Transaction Amount: 693454.45


## A5 – Final Date Range Check

This test confirms that the transaction data covers the expected analysis period and that no records fall outside the intended date range.

In [8]:
transactions["Transaction_Date"] = pd.to_datetime(
    transactions["Transaction_Date"]
)

print("Earliest Transaction Date:",
      transactions["Transaction_Date"].min())

print("Latest Transaction Date:",
      transactions["Transaction_Date"].max())

Earliest Transaction Date: 2026-01-01 00:00:00
Latest Transaction Date: 2026-03-31 00:00:00


## A6 – Final KPI Accuracy Check

This test recalculates the core FinTrust KPIs directly from the final cleaned datasets to confirm that the values used in SQL, Python and Power BI remain accurate and consistent.

In [9]:
# Recalculate core KPIs

total_customers = customers["Customer_ID"].nunique()

total_transactions = transactions["Transaction_ID"].nunique()

total_transaction_value = transactions["Amount_NGN"].sum()

average_transaction_value = transactions["Amount_NGN"].mean()

success_rate = (
    (transactions["Transaction_Status"] == "Successful").mean() * 100
)

risk_review_rate = (
    (transactions["Risk_Review_Flag"] == "Yes").mean() * 100
)

# Display results
print("Total Customers:", total_customers)
print("Total Transactions:", total_transactions)
print("Total Transaction Value: NGN {:,.2f}".format(total_transaction_value))
print("Average Transaction Value: NGN {:,.2f}".format(average_transaction_value))
print("Transaction Success Rate: {:.2f}%".format(success_rate))
print("Risk Review Rate: {:.2f}%".format(risk_review_rate))

Total Customers: 1500
Total Transactions: 12000
Total Transaction Value: NGN 560,477,354.85
Average Transaction Value: NGN 46,706.45
Transaction Success Rate: 90.47%
Risk Review Rate: 19.60%


## A7 – SQL Verification

### A7.1 – Monthly Transaction Trend Verification

This test recalculates the monthly transaction totals in Python and compares them with the final SQL analysis to confirm that the SQL results remain accurate.

In [10]:
monthly_check = (
    transactions
    .groupby(transactions["Transaction_Date"].dt.to_period("M"))
    .agg(
        Total_Transactions=("Transaction_ID", "count"),
        Total_Transaction_Value=("Amount_NGN", "sum")
    )
    .reset_index()
)

monthly_check["Total_Transaction_Value"] = (
    monthly_check["Total_Transaction_Value"].round(2)
)

monthly_check

,Transaction_Date,Total_Transactions,Total_Transaction_Value
0,2026-01,4133,1.884894e+08
1,2026-02,3734,1.757869e+08
2,2026-03,4133,1.962010e+08


### Result

The Python recalculation matched the final SQL monthly transaction results for January, February and March 2026.

**Conclusion:** The monthly SQL analysis is accurate and consistent with the final cleaned transaction dataset.

### A7.2 – Risk Review by Transaction Amount Verification

This test recalculates the risk-review rate across transaction amount bands in Python and compares the results with the final SQL analysis.

In [11]:
transactions["Amount_Band"] = pd.cut(
    transactions["Amount_NGN"],
    bins=[0, 10000, 50000, 100000, 200000, float("inf")],
    labels=[
        "Below 10K",
        "10K - 49,999",
        "50K - 99,999",
        "100K - 199,999",
        "200K and Above"
    ],
    right=False
)

risk_check = (
    transactions
    .groupby("Amount_Band", observed=False)
    .agg(
        Total_Transactions=("Transaction_ID", "count"),
        Risk_Review_Transactions=(
            "Risk_Review_Flag",
            lambda x: (x == "Yes").sum()
        )
    )
    .reset_index()
)

risk_check["Risk_Review_Rate_Percent"] = (
    risk_check["Risk_Review_Transactions"]
    / risk_check["Total_Transactions"]
    * 100
).round(2)

risk_check

,Amount_Band,Total_Transactions,Risk_Review_Transactions,Risk_Review_Rate_Percent
0,Below 10K,5947,1050,17.66
1,"10K - 49,999",3110,554,17.81
2,"50K - 99,999",1214,230,18.95
3,"100K - 199,999",994,220,22.13
4,200K and Above,735,298,40.54


### Result

The Python recalculation matched the final SQL risk-review results across all transaction amount bands.

**Conclusion:** The SQL analysis showing higher risk-review rates for larger transaction values is accurate and reproducible.

### A7.3 – Failure Rate Verification

This test recalculates transaction failure rates by banking channel and transaction type in Python and compares the results with the final SQL analysis.

In [12]:
failure_check = (
    transactions
    .groupby(["Channel", "Transaction_Type"])
    .agg(
        Total_Transactions=("Transaction_ID", "count"),
        Failed_Transactions=(
            "Transaction_Status",
            lambda x: (x == "Failed").sum()
        )
    )
    .reset_index()
)

failure_check["Failure_Rate_Percent"] = (
    failure_check["Failed_Transactions"]
    / failure_check["Total_Transactions"]
    * 100
).round(2)

failure_check.sort_values(
    "Failure_Rate_Percent",
    ascending=False
).head(10)

,Channel,Transaction_Type,Total_Transactions,Failed_Transactions,Failure_Rate_Percent
13,POS,Bill Payment,291,21,7.22
23,USSD,Transfer,250,18,7.20
16,POS,Deposit,268,17,6.34
25,Web,Bill Payment,223,14,6.28
24,Web,Airtime/Data,176,11,6.25
8,Mobile App,Card Purchase,1263,78,6.18
11,Mobile App,Transfer,1526,93,6.09
21,USSD,Cash Withdrawal,100,6,6.00
9,Mobile App,Cash Withdrawal,626,36,5.75
7,Mobile App,Bill Payment,643,37,5.75


## A8 – Python Reproducibility Check

This test confirms that the final Python notebook can be restarted and executed from the cleaned source datasets without errors, and that the analytical outputs remain consistent.

### Result

The notebook was restarted and executed from beginning to end using the final cleaned FinTrust datasets.

All cells executed successfully without errors, and the main analytical outputs remained consistent with the previously validated SQL and Power BI results.

**Conclusion:** The final Python analysis is reproducible.